In [1]:
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm
import math
import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

In [ ]:
def split_model():
    device_map = {}
    world_size = torch.cuda.device_count()
    num_layers = 32
    num_layers_per_gpu = math.ceil(num_layers / (world_size - 0.5))
    num_layers_per_gpu = [num_layers_per_gpu] * world_size
    num_layers_per_gpu[0] = math.ceil(num_layers_per_gpu[0] * 0.5)
    layer_cnt = 0
    for i, num_layer in enumerate(num_layers_per_gpu):
        for j in range(num_layer):
            device_map[f'language_model.model.layers.{layer_cnt}'] = i
            layer_cnt += 1
    device_map['vision_model'] = 0
    device_map['mlp1'] = 0
    device_map['language_model.model.tok_embeddings'] = 0
    device_map['language_model.model.embed_tokens'] = 0
    device_map['language_model.output'] = 0
    device_map['language_model.model.norm'] = 0
    device_map['language_model.lm_head'] = 0
    device_map[f'language_model.model.layers.{num_layers - 1}'] = 0

    return device_map

In [2]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values


In [3]:
model_name = 'internvl'
prompt_type = 'zeroshot'

In [4]:
dataset = 'type1'
image_type = 'original'
device = "cuda:1"

In [ ]:
qas = pd.read_json(f'../{dataset}/qa.json')
qas

In [6]:
image_indices = qas['image_index'].values.astype(int)
questions = qas['question'].values
answers = qas['answer'].values

In [7]:
image_base_path = f'../{dataset}/{image_type}/'

In [ ]:
all_images = os.listdir(image_base_path)
index_to_image = {}

if dataset == 'type1':
    prefix = 'multichart_'
    if image_type == 'original' or image_type == 'simple':
        sep = '_'
    else:
        sep = '.'

for index in set(image_indices):
    for image in all_images:
        if image.startswith(f'{prefix}{index}{sep}'):
            if index not in index_to_image:
                index_to_image[index] = []
            index_to_image[index].append(image)

index_to_image

In [9]:
def get_message(image_index, prompt, question):
    pixel_values = None
    num_patches_list = []
    for image in index_to_image[image_index]:
        image_path = f'{image_base_path}{image}'
        image_tensor = load_image(image_path, max_num=12).to(torch.bfloat16).to(device)
        if pixel_values is None:
            pixel_values = image_tensor
        else:
            pixel_values = torch.cat((pixel_values, image_tensor), dim=0)
        num_patches_list.append(image_tensor.size(0))
    
    img_prefix = '<image>\n' * len(index_to_image[image_index])
    message = f'{img_prefix}{prompt.format(question)}'
    return message, pixel_values, num_patches_list

In [ ]:
path = 'OpenGVLab/InternVL2-8B'
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True,
    cache_dir = '/media/vivek/drive/multichartqa/models_cache',
    device_map=device
    ).eval()
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False)
generation_config = dict(max_new_tokens=1024, do_sample=True)

In [11]:
model_responses = []

In [12]:
def run_model():
    for i, question in enumerate(tqdm(questions[:5])):
        image_index = image_indices[i]
        prompt = """Your task is the answer the question based on the given image. Your final answer to the question should strictly be in the format - "Final Answer:" <final_answer>.\n\nQuestion: {}""" 
        message, pixel_values, num_patches_list = get_message(image_index, prompt, question)
        response = model.chat(tokenizer, pixel_values, message, generation_config, num_patches_list=num_patches_list)
        
        model_responses.append({'question_id': i, 'question': question, 'gold': answers[i], 'response': response.strip()})
        print(f'Question: {question}\nGold: {answers[i]}\nResponse: {response.strip()}\n\n')

In [ ]:
run_model()

In [14]:
# saving the model responses
os.makedirs(f'../model_responses/{dataset}', exist_ok=True)

model_responses_df = pd.DataFrame(model_responses)
model_responses_df.to_json(f'../model_responses/{dataset}/{model_name}_{image_type}_{prompt_type}.json', orient='records')